In [1]:
import numpy as np
import h5py
import time
import random
from tqdm import tqdm
import os
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm

# ==================== 0. 字体配置 ====================
def set_chinese_font():
    try:
        font_paths = ['/usr/share/fonts/truetype/wqy/wqy-microhei.ttc',
                      'C:/Windows/Fonts/simhei.ttf',
                      '/System/Library/Fonts/PingFang.ttc']
        for font_path in font_paths:
            if os.path.exists(font_path):
                fm.fontManager.addfont(font_path)
                plt.rcParams['font.family'] = fm.FontProperties(fname=font_path).get_name()
                plt.rcParams['axes.unicode_minus'] = False
                return True
    except: pass
    return False
set_chinese_font()

# ==================== 1. 卫星抗干扰环境 (保持一致) ====================
class SatelliteEnvV3:
    def __init__(self, h5_path):
        print(f"📂 正在预载入数据集: {os.path.basename(h5_path)} ...")
        with h5py.File(h5_path, 'r') as f:
            self.pred_map = f['Y_horizon'][:] 
            self.truth_map = f['Y_horizon'][:]
            self.type_data = f['gt_type'][:]
            
        self.num_channels = 10
        self.max_steps = len(self.pred_map)
        self.current_step = 0
        self.last_action = 0

    def reset(self):
        self.current_step = 0
        self.last_action = random.randint(0, 9)
        return self._get_state()

    def _get_state(self):
        # Q-Learning 状态离散化处理
        # 取当前步长预测图中能量最高的信道索引作为“状态”
        map_feat = self.pred_map[self.current_step, 0] # 取第1步预测
        dominant_channel = np.argmax(map_feat)
        interference_type = int(self.type_data[self.current_step])
        # 组合成一个唯一的整数状态 ID (0-49)
        state_id = dominant_channel * 5 + interference_type
        return state_id

    def step(self, action):
        is_collision = self.truth_map[self.current_step, 0, action] > 0.5
        future_risk = np.mean(self.pred_map[self.current_step, :, action])
        
        if is_collision:
            reward = -100.0
        else:
            reward = 15.0 - (future_risk * 30.0)
            if action == self.last_action:
                reward += 5.0 
            else:
                reward -= 2.0 
            
        self.last_action = action
        self.current_step += 1
        done = self.current_step >= self.max_steps - 1
        next_state = self._get_state() if not done else -1
        return next_state, reward, done, is_collision

# ==================== 2. Q-Learning 智能体 ====================
class QLearningAgent:
    def __init__(self, state_dim=50, action_dim=10):
        # 初始化 Q 表: 行代表状态，列代表动作
        self.q_table = np.zeros((state_dim, action_dim))
        self.lr = 0.1
        self.gamma = 0.98
        self.epsilon = 1.0
        self.epsilon_min = 0.05
        self.epsilon_decay = 0.995

    def choose_action(self, state):
        start_time = time.time()
        if random.random() <= self.epsilon:
            action = random.randint(0, 9)
        else:
            action = np.argmax(self.q_table[state])
        latency = time.time() - start_time
        return action, latency

    def learn(self, s, a, r, ns, done):
        if done:
            target = r
        else:
            target = r + self.gamma * np.max(self.q_table[ns])
        
        # Q-Learning 更新公式: Q(s,a) = Q(s,a) + lr * (target - Q(s,a))
        self.q_table[s, a] += self.lr * (target - self.q_table[s, a])
        
        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay

# ==================== 3. 训练执行与指标报告 ====================
def run_q_learning_training_with_metrics():
    DATA_PATH = "/root/autodl-tmp/validate/0218/Prediction/sim_dataset_v6_5/academic_long_horizon_v6_5_200k.h5"
    env = SatelliteEnvV3(DATA_PATH)
    agent = QLearningAgent(state_dim=50, action_dim=10) # 10信道*5种类
    
    episodes = 100
    metrics = {'reward': [], 'sr': [], 'hops': [], 'latency': [], 'throughput': [], 'stability': []}
    convergence_ep = -1

    print("🚀 启动 Q-Learning 离线对比训练 (作为基础 Baseline)...")
    for ep in range(episodes):
        state = env.reset()
        ep_reward, collisions, steps, hops = 0, 0, 0, 0
        ep_latencies, actions = [], []

        pbar = tqdm(total=env.max_steps, desc=f"Ep {ep+1}/{episodes}", leave=False)
        while True:
            action, lat = agent.choose_action(state)
            ep_latencies.append(lat)
            actions.append(action)

            if steps > 0 and action != env.last_action:
                hops += 1
                
            next_state, reward, done, collision = env.step(action)
            
            # 训练 Q-Learning
            if not done:
                agent.learn(state, action, reward, next_state, done)
                
            state = next_state
            ep_reward += reward
            if collision: collisions += 1
            steps += 1
            pbar.update(1)
            if done: break
        
        pbar.close()
        
        # 指标计算
        sr = (1 - collisions / steps) * 100
        avg_hops = hops / (steps / 100)
        avg_lat = np.mean(ep_latencies) * 1000
        throughput = (1 - (collisions / steps)) * 1.0
        stability = np.std(actions)

        if convergence_ep == -1 and sr >= 95.0: convergence_ep = ep + 1

        metrics['reward'].append(ep_reward)
        metrics['sr'].append(sr)
        metrics['hops'].append(avg_hops)
        metrics['latency'].append(avg_lat)
        metrics['throughput'].append(throughput)
        metrics['stability'].append(stability)

        print(f"✅ Ep {ep+1} | 成功率: {sr:.2f}% | 每百步跳频: {avg_hops:.1f} | 延迟: {avg_lat:.2f}ms | 吞吐量: {throughput:.2f}")

    print("\n" + "="*50)
    print("📊 Q-Learning 离线实验对比报告总结")
    print("-" * 50)
    print(f"1. 收敛轮次 (Convergence Episode): {convergence_ep if convergence_ep != -1 else '未收敛'}")
    print(f"2. 平均推理时延 (Inference Latency): {np.mean(metrics['latency']):.4f} ms")
    print(f"3. 避障成功率峰值 (Peak Success Rate): {np.max(metrics['sr']):.2f} %")
    print(f"4. 稳态跳频代价 (Avg Switching Cost): {np.mean(metrics['hops'][-10:]):.2f} hops/100steps")
    print(f"5. 归一化吞吐量 (Throughput): {np.mean(metrics['throughput'][-10:]):.4f}")
    print(f"6. 动作稳定性 (Policy Stability Std): {np.mean(metrics['stability'][-10:]):.4f}")
    print("="*50)

    # 保存 Q 表
    np.save("q_table_offline.npy", agent.q_table)
    return metrics

if __name__ == "__main__":
    run_q_learning_training_with_metrics()

📂 正在预载入数据集: academic_long_horizon_v6_5_200k.h5 ...
🚀 启动 Q-Learning 离线对比训练 (作为基础 Baseline)...


✅ Ep 1 | 成功率: 85.94% | 每百步跳频: 49.9 | 延迟: 0.00ms | 吞吐量: 0.86


✅ Ep 2 | 成功率: 86.82% | 每百步跳频: 50.4 | 延迟: 0.00ms | 吞吐量: 0.87


✅ Ep 3 | 成功率: 87.15% | 每百步跳频: 47.3 | 延迟: 0.00ms | 吞吐量: 0.87


✅ Ep 4 | 成功率: 88.11% | 每百步跳频: 48.6 | 延迟: 0.00ms | 吞吐量: 0.88


...

✅ Ep 96 | 成功率: 88.36% | 每百步跳频: 46.8 | 延迟: 0.00ms | 吞吐量: 0.88


✅ Ep 97 | 成功率: 87.65% | 每百步跳频: 45.7 | 延迟: 0.00ms | 吞吐量: 0.88


✅ Ep 98 | 成功率: 87.88% | 每百步跳频: 47.6 | 延迟: 0.00ms | 吞吐量: 0.88


✅ Ep 99 | 成功率: 88.08% | 每百步跳频: 46.3 | 延迟: 0.00ms | 吞吐量: 0.88


✅ Ep 100 | 成功率: 87.88% | 每百步跳频: 44.9 | 延迟: 0.00ms | 吞吐量: 0.88

📊 Q-Learning 离线实验对比报告总结
--------------------------------------------------
1. 收敛轮次 (Convergence Episode): 未收敛
2. 平均推理时延 (Inference Latency): 0.0043 ms
3. 避障成功率峰值 (Peak Success Rate): 88.46 %
4. 稳态跳频代价 (Avg Switching Cost): 46.23 hops/100steps
5. 归一化吞吐量 (Throughput): 0.8795
6. 动作稳定性 (Policy Stability Std): 2.6245
